# 微調對話式ChatGPT語言模型

Finetune Dialog LLM

使用Langbot bloom裁切小模型做展示 ('YeungNLP/bloomz-396m-zh'小模型也可以，過程一樣)

Colab T4 可以在不到兩小時之內訓練一個epoch成功!

只要一個epoch效果就很不錯!


    第1種小模型: 瀾舟

        model = AutoModelForCausalLM.from_pretrained('Langboat/bloom-389m-zh',torch_dtype='auto')

        https://huggingface.co/Langboat

    第2種小模型: 原始模型式float16，模型更小

        tokenizer = BloomTokenizerFast.from_pretrained('YeungNLP/bloomz-396m-zh')
        model = AutoModelForCausalLM.from_pretrained('YeungNLP/bloomz-396m-zh',torch_dtype='auto')
        
資料集必須使用相同的tokenizer先處理。參看資料集前處理步驟之程式碼!

In [ ]:
# 選用幾個GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

# Install packages

In [ ]:
#!pip install transformers==4.32
#!pip install datasets
#!pip install accelerate -U

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/LLM

/content/drive/MyDrive/LLM


# Libraries

In [1]:
import torch
import transformers
from transformers import AutoModelForCausalLM, BloomTokenizerFast, BloomForCausalLM, TrainingArguments

import datasets

# Load model and tokenizer

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("gpu")

## Load base model

In [2]:
# initial base pretrained model YeungNLP/bloomz-396m-zh
# 第1種小模型: 原始模型式float32, 1.65GB
# model = AutoModelForCausalLM.from_pretrained('Langboat/bloom-389m-zh',torch_dtype='auto')
model = AutoModelForCausalLM.from_pretrained('YeungNLP/bloomz-396m-zh',torch_dtype='auto')


In [ ]:
# 當GPU記憶體很有限時，可以降低GPU記憶體的需求，但執行時間會拉長些
# model.gradient_checkpointing_enable()

## Load tokenizer

In [3]:
# 第1種小模型: 瀾舟
tokenizer = BloomTokenizerFast.from_pretrained('YeungNLP/bloomz-396m-zh')

## eos_token與pad_token如何設計?

    tokenizer的特殊token是否要額外做設定?

    Bloom都已經設定好了，不必去更動。 (llama必須特別去設定eos_token)

    padding_side內定靠左，不必更動。(llama必須靠右)

In [ ]:
# 不需要做以下特別設定
# tokenizer.add_special_tokens({"pad_token": tokenizer.unk_token})
# tokenizer.padding_side = "left"  # Allow batched inference

In [4]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>'}

# Load training dataset

In [5]:
train_data_path = "./train_dataset_YeungNLP簡體"
train_data = datasets.load_from_disk(train_data_path+'/data_train/')
val_data = datasets.load_from_disk(train_data_path+'/data_val/')

In [6]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 15786
})

In [7]:
val_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 241
})

In [8]:
tokenizer.decode(train_data[4494]['input_ids'])

'Human: \n请判断以下文字的属于何种面向类别，面向类别有整洁舒适、设施、服务、地点、性价比、其他；并判断以下文字属于此面向的何种情绪类别，类别有正面、负面、中立、无情绪 晚餐如果能在门口有明确告示今日套餐会更好，客人可以更清楚晚餐的用餐方式。地点、设备、房间大小都非常的棒\n\nAssistant: \n这段文字是属于:设施面向,服务面向,地点面向,设施情绪: 正面 服务情绪: 负面 地点情绪: 正面 整体情绪: 正面</s>'

# Data collator

把訓練資練成批，每批長度相同(以長度最大者為依據)，不足長度者塞入pad填充(Bloom填充塞到左側，與Llama不同)

In [ ]:
#  label_pad_token_id (int, *optional*, defaults to -100):
data_collator = transformers.DataCollatorForSeq2Seq(tokenizer,
                                             return_tensors="pt",
                                             padding=True)

In [ ]:
data_collator

DataCollatorForSeq2Seq(tokenizer=BloomTokenizerFast(name_or_path='Langboat/bloom-389m-zh', vocab_size=42437, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False), model=None, padding=True, max_length=None, pad_to_multiple_of=None, label_pad_token_id=-100, return_tensors='pt')

# 設定訓練參數

In [ ]:
# https://discuss.huggingface.co/t/what-is-the-purpose-of-use-cache-in-decoder/958/3
# model.config.use_cache = False  # 內定值，應該不必設定

In [ ]:
BATCH_SIZE = 128
MICRO_BATCH_SIZE = 10  # T5 GPU 12->11.6GB還不會Out of memory
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE

training_args = TrainingArguments(


    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    fp16=True, # This is for GPU not for CPU
    #fp16=False,
    optim='adamw_torch',



    # Belle
    warmup_ratio=0.05,
    weight_decay=0.00001, # Belle
    learning_rate=8e-6, # Belle # 是否太小?收斂慢?

    # 另外一種設定可試試看
    #warmup_steps=100,
    #weight_decay=0.001,
    #learning_rate=2e-5, #
    #learning_rate=5e-4, #稍大

    output_dir="my-checkpoints",
    report_to='none',

    num_train_epochs=3, # 訓練回合

    save_strategy='steps',
    evaluation_strategy='steps',

    logging_steps= 20,
    save_steps=20,  # 執行多少批次就存一次檔案
    save_total_limit=2, # 最多只存目前與前一次兩個

    overwrite_output_dir=True,
    # load_best_model_at_end=True,

)

# 初始化Trainer訓練模型
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=training_args,
    data_collator=data_collator,
)

# 開始訓練

In [ ]:
%%time

# 從新開始訓練
trainer.train()

# 持續訓練，若被中斷執行，也沒在怕! 只要設定resume_from_checkpoint=True
# trainer.train(resume_from_checkpoint=True)

You're using a BloomTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss


CPU times: user 1.61 s, sys: 3.54 s, total: 5.15 s
Wall time: 22.1 s


TrainOutput(global_step=640, training_loss=0.0, metrics={'train_runtime': 0.1745, 'train_samples_per_second': 90488.793, 'train_steps_per_second': 750.921, 'total_flos': 2.2370468957159424e+16, 'train_loss': 0.0, 'epoch': 4.86})

T5 訓練1 epoch花費1.5小時左右

    epoch 1:

    360	1.718900	1.728449
    CPU times: user 1h 5min 49s, sys: 14min 12s, total: 1h 20min 2s
    Wall time: 1h 39min
    TrainOutput(global_step=362, training_loss=1.8476548886430857, metrics={'train_runtime': 5940.6456, 'train_samples_per_second': 7.807, 'train_steps_per_second': 0.061, 'total_flos': 2.131411554926592e+16, 'train_loss': 1.8476548886430857, 'epoch': 1.0})

    epoch 2: (有over training過度訓練了! 但是模型給出的答案比較好!!)
    360	1.647100	1.728379
    CPU times: user 1h 5min 22s, sys: 14min 4s, total: 1h 19min 26s
    Wall time: 1h 40min 44s
    TrainOutput(global_step=362, training_loss=1.6298396752025541, metrics={'train_runtime': 6044.9079, 'train_samples_per_second': 7.672, 'train_steps_per_second': 0.06, 'total_flos': 2.129500450081997e+16, 'train_loss': 1.6298396752025541, 'epoch': 1.0})

# Save model

In [ ]:
model.save_pretrained('my-pretrained-3epochs-YeungNLP-zh-ch')

# References

    有很多方法可以避免顯存不足以及訓練時間過長的方法，包括以下幾種方法：

    梯度累積（Gradient Accumulation）
    凍結（Freezing）
    自動混合精度（Automatic Mixed Precision）
    8位優化器（8-bit Optimizers）
    梯度檢查點（Gradient Checkpointing）
    快速分詞器（Fast Tokenizers）
    動態填充（Dynamic Padding）
    均勻動態填充（Uniform Dynamic Padding）
    其中1-5是神經網路通用的方法，可以用在任何網路的性能優化上，6-8是針對nlp領域的性能優化方法。
    https://zhuanlan.zhihu.com/p/555283334


    https://github.com/huggingface/transformers/blob/main/src/transformers/training_args.py
    https://github.com/huggingface/transformers/blob/main/src/transformers/data/data_collator.py
    https://github.com/huggingface/transformers/blob/main/src/transformers/trainer.py
    https://www.deepspeed.ai/docs/config-json/
    https://huggingface.co/docs/accelerate/usage_guides/deepspeed
    https://huggingface.co/transformers/v4.10.1/main_classes/deepspeed.html
    https://github.com/tatsu-lab/stanford_alpaca/issues/176